# Updating data from GenBank

Author: Alexander Maksiaev

Purpose: Update labels from previously gotten data from GISAID + Andersen, using Genbank.

In [1]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

date_range = "01-01-2024--04-14-2025"
update_date = "05-21-2025"

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# originals = downloads + "GISAID_Andersen_Combined_Files/"
# temp_files = downloads + "Andersen_Temp_Files/"
# github_files = downloads + "Andersen_Downloads/avian-influenza/metadata/"
# complete = originals + date_range + "_B3_13_D1_1/D1_1/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Combinations/GISAID_Andersen/B3_13_D1_1/" 
temp_files = downloads + "Andersen/temp/"
github_files = downloads + "Andersen/avian-influenza/metadata/"
complete = originals + "01-01-2024--04-14-2025_B3_13_D1_1/D1_1/"
# complete = originals + "11-2023--04-14-2025_B3_13/"



os.chdir(complete)

## Collection Dates

In [2]:
# Upload saved data 
# os.chdir(temp_files + "saved/")
os.chdir(downloads)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

os.chdir(github_files)
metadata = pd.read_csv("SraRunTable_automated.csv") 
metadata = metadata.merge(metadata_normalized, how="outer")

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
metadata_genbank = metadata.merge(genbank_mapping, how="outer") #, on="Run")
metadata_genbank = metadata_genbank[metadata_genbank["is_retracted"] == False]
# metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv") # Since 1/1/2024
os.chdir(originals)

display(metadata_genbank)
print(metadata_genbank["Sample Name"])

metadata_genbank.to_csv("metadata_genbank_5_22_2025.csv")

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,BioSample Accession,is_retracted,retraction_detection_date_utc,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,SRS17903639,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,SRS17903639,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,SRS17903636,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,SRS17903636,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,SRS17903631,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93753,SRR33565489,WGS,148.31,58438942,PRJNA1102327,SAMN48491499,Viral,22852781,USDA-NVSL,2025,...,SRS25033260,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93754,SRR33565490,WGS,148.05,73125034,PRJNA1102327,SAMN48491498,Viral,28848992,USDA-NVSL,2025,...,SRS25033259,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93755,SRR33565490,WGS,148.05,73125034,PRJNA1102327,SAMN48491498,Viral,28848992,USDA-NVSL,2025,...,SRS25033259,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93756,SRR33565491,WGS,146.78,74601613,PRJNA1102327,SAMN48491497,Viral,29227804,USDA-NVSL,2025,...,SRS25033258,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


0        22-005893-001
1        22-005893-001
2            AH0210318
3            AH0210318
4        22-005158-001
             ...      
93753    25-013627-005
93754    25-013627-004
93755    25-013627-004
93756    25-013627-002
93757    25-013627-002
Name: Sample Name, Length: 88180, dtype: object


In [3]:
# Get geolocation for second state attribute

os.chdir(home + "/references/")
state_ref = pd.read_csv("states_ref.csv")
metadata_genbank["name_state"] = metadata_genbank["genbank_name"].apply(lambda x: x.split("/")[2].replace("_", " ")) # Get the name of the state
metadata_genbank["Geo_Location"] = metadata_genbank["name_state"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] + "-" + x if x in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x, 'Abbreviation'].iloc[0] if x in state_ref["State"].values else x)

print(metadata_genbank["Geo_Location"])

AttributeError: 'float' object has no attribute 'split'

In [ ]:
# no_updates = pd.DataFrame()
# no_updates_isolate = []

# Get only labels that have no states or collection dates, and update them

def update(file_name, update_date):
    updates = {}
    with open(file_name) as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if line[0] == ">": # It's a header
                header = line 
                collection_date = header.split("|")[-3]
                state = header.split("/")[2].replace("_", " ")
                header = header.replace(header.split("|")[-4] + "|", "") # remove previous state
                # state = state.replace(": ", "-")
                # geo_location_collection_date = state + "|" + collection_date
                # geo_location_collection_date = collection_date

                # partial = value.split("_")[-1] # If 25_, get the last bit
                # print(partial)
                
                
                isolate = header.split("/")[3]
                sequence = lines[i + 1] # Sequence always comes in one line after header

                # if len(row) < 1: # If there is no isolate that we know of as-is
                digits = isolate.split("-")
                built_isolate = ""
                # other = ""
                for d in digits:
                    # print(d)
                    if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
                        built_isolate = d + "-"
                    elif len(d) == 3 and d.isnumeric():
                        built_isolate = built_isolate + d
                    # elif d.isnumeric() == False: # If it's a weird isolate
                    #     other = d + "-"
                    # else: # If it's a weird isolate
                    #     other = other + d
                # Now add to list to check in Andersen files without doing wild for loops
                if len(built_isolate) == 10: # If this is a correctly formatted isolate
                    # isolates.append(isolate)
                    # All headers are followed by sequences
                    isolate = built_isolate

                row = metadata_genbank[metadata_genbank["genbank_name"].str.contains(isolate)]

                try:
                    row = row.values[-1]
                    print(row)
                except:
                    # row = metadata_genbank.loc[metadata_genbank["Sample Name"].str.contains(isolate), :]
                    print(isolate)


                # print(row)
                # break 
                    

                # if "-" not in collection_date: # If there are no dashes, i.e. if it's just the year
                #     # Find the correct collection date, if it exists
                
                try: 
                    collection_date = row["Collection_Date"].values[0]
                    if "-" in collection_date:
                        print(collection_date)
                #     print(id)
                #     geo_location_collection_date = search_collection_date(id, row) # Update unknown dates, if possible
                except:
                    print("No date found for isolate", isolate)
                    #         # no_updates_isolate.append(isolate)

                # if state == "USA": # If we don't have a state

                # try: 
                # state_new = row["Geo_Location"]
                try:
                    state = row["Geo_Location"].values[0]
                    print(state)
                except:
                    state = str([state_ref.loc[state_ref["Abbreviation"] == state, 'Country'].iloc[0] + "-" + state if state in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == state, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == state, 'Abbreviation'].iloc[0] if state in state_ref["State"].values else state][0])
                    
                    # no_updates_isolate.append(isolate)

                updates[header] = [state, collection_date, sequence]

        f.close()

    updates_df = pd.DataFrame.from_dict(updates, orient="index", columns=["state", "collection_date", "sequence"])
    updates_df["geo_location_collection_date"] = updates_df["state"] + "|" + updates_df["collection_date"]
    updates_df["header"] = updates_df.index
    updates_df = updates_df.reset_index()

    updated_file_name = ".".join(file_name.split(".")[:-1]) + "_" + date_range + "_" + update_date + "_update." + file_name.split(".")[-1]

    with open(updated_file_name, "w") as g:

        for i, row in updates_df.iterrows():
            header = row["header"]
            # print(header)
            # print(header.split("|")[-3])
            
            # if header.split("/")[2] == "USA":
            #     header = header.replace(header.split("/")[2], row["state"])
            
            # header = header.replace(str(header.split("|")[-3]), str(row["geo_location_collection_date"])) # Only the first instance is replaced
            # header = str(row["collection_date"]).join(header.rsplit(str(header.split("|")[-3]), 1))
            header = str(row["geo_location_collection_date"]).join(header.rsplit(str(header.split("|")[-3]), 1))
            g.write(header)
            g.write(row["sequence"])

        g.close()

    # no_updates["isolate"] = no_updates_isolate
    # no_updates.to_csv("not_updated.csv")


In [ ]:
# Create files with updates

# os.chdir(originals)

# for dirpath, dirs, files in os.walk(originals + date_range + "_B3_13_D1_1/"): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         update(file_name, update_date)
#     break 

os.chdir(complete)
# file = "8-newid_B3.13_APR14_NS_trim_codon_aln_n3471_FINAL.fasta"
# update(file, update_date)

for dirpath, dirs, files in os.walk(complete): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file)
        update(file_name, update_date)
    break 

002460-001
No date found for isolate 002460-001
002634-002
No date found for isolate 002634-002
['SRR32416423' 'WGS' 147.47 52603175 'PRJNA1207547' 'SAMN46923609' 'Viral'
 20258400 'USDA-NVSL' '2025-01-15' 'public' 'run,run.zq,fastq' 's3,gs'
 's3.us-east-1,gs.us-east1' 'SRX27745586' 'USA' nan 'USA/Maryland//' nan
 'NextSeq 2000' '25-003141-001' '25-003141-001-original' 'PAIRED' 'RANDOM'
 'VIRAL RNA' 'Influenza A virus' 'ILLUMINA' '2025-02-21 00:32:36'
 '2025-02-20 13:45:38' 1 '25-003141-001' 'SRP557452' nan '/' 'SRS24135368'
 False nan 'SRR32416423_PB2_cns.fa'
 'Consensus_SRR32416423_PB2_cns_threshold_0.5_quality_20' 'SRR32416423'
 'PB2' 'PV336953.1' 1 'A/Mallard/MD/25-003141-001-original/2025' 'MD'
 'USA-MD']
No date found for isolate 141
002453-003
No date found for isolate 002453-003
005883-002
No date found for isolate 005883-002
AIVPHL-2662
No date found for isolate AIVPHL-2662
AIVPHL-2850
No date found for isolate AIVPHL-2850
['SRR32416478' 'WGS' 146.46 94722578 'PRJNA1207547' 'S

In [ ]:
# search_collection_date_term("PP752829.1", metadata_genbank)